# Clase 203 — Reentrenamiento programado (Prefect)

Pipeline `pull → validate → train → evaluate → champion-challenger` con Prefect (más liviano que Airflow para una clase).

Requiere: `pip install prefect mlflow scikit-learn`. Para correr en cloud: `prefect cloud login`.

## Setup

In [ ]:
import os, tempfile, shutil
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'retrain_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

import mlflow
mlflow.set_tracking_uri(f'file:{WORK}/mlruns')
mlflow.set_experiment('continual-training')

## 1. Tareas individuales (idempotentes)

In [ ]:
from prefect import flow, task, get_run_logger
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import joblib, json, numpy as np, hashlib

@task(retries=2, retry_delay_seconds=10)
def pull_data(execution_date: str):
    log = get_run_logger()
    X, y = load_iris(return_X_y=True)
    # Simular drift incremental: inyectar ruido proporcional al día
    seed = int(hashlib.md5(execution_date.encode()).hexdigest(), 16) % 1000
    rng = np.random.default_rng(seed)
    X = X + rng.normal(0, 0.1, X.shape)
    log.info(f'pulled {len(X)} rows for {execution_date}')
    return X, y

@task
def validate_data(X, y):
    log = get_run_logger()
    assert not np.isnan(X).any(), 'NaN in features'
    assert set(np.unique(y)) == {0, 1, 2}, 'unexpected labels'
    log.info('data validation passed')

@task
def train(X, y, execution_date: str):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
    with mlflow.start_run(run_name=f'train_{execution_date}') as run:
        m = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, ytr)
        f1 = f1_score(yte, m.predict(Xte), average='macro')
        mlflow.log_metric('f1', f1)
        mlflow.log_param('execution_date', execution_date)
        mlflow.sklearn.log_model(m, name='model')
        return {'run_id': run.info.run_id, 'f1': f1}

@task
def promote_if_better(challenger: dict, alias='champion', delta=0.005):
    log = get_run_logger()
    client = mlflow.tracking.MlflowClient()
    # Buscar champion actual
    try:
        champ_mv = client.get_model_version_by_alias('iris-prod', alias)
        champ_run = client.get_run(champ_mv.run_id)
        champ_f1 = champ_run.data.metrics.get('f1', 0)
    except Exception:
        champ_f1 = 0
        log.info('no champion yet — promoting first')

    if challenger['f1'] > champ_f1 + delta:
        mv = mlflow.register_model(f"runs:/{challenger['run_id']}/model", 'iris-prod')
        client.set_registered_model_alias('iris-prod', alias, mv.version)
        log.info(f'PROMOTED v{mv.version} (f1 {challenger["f1"]:.4f} > champion {champ_f1:.4f})')
        return {'promoted': True, 'version': mv.version}
    log.info(f'SKIPPED (challenger {challenger["f1"]:.4f} not > champion {champ_f1:.4f} + {delta})')
    return {'promoted': False}

## 2. Flow (el DAG)

In [ ]:
@flow(name='daily-retrain')
def daily_retrain(execution_date: str):
    X, y = pull_data(execution_date)
    validate_data(X, y)
    challenger = train(X, y, execution_date)
    return promote_if_better(challenger)

# Correr 5 días simulados
for day in ['2026-06-01', '2026-06-02', '2026-06-03', '2026-06-04', '2026-06-05']:
    print(f'\n=== {day} ===')
    result = daily_retrain(day)
    print(result)

## 3. Estado final del Model Registry

In [ ]:
client = mlflow.tracking.MlflowClient()
for mv in client.search_model_versions("name='iris-prod'"):
    aliases = ','.join(mv.aliases) if mv.aliases else '-'
    print(f'v{mv.version}: aliases={aliases}, run={mv.run_id[:8]}')

champ = client.get_model_version_by_alias('iris-prod', 'champion')
print(f'\nCHAMPION: v{champ.version}')

## 4. Schedule + deploy (Prefect Cloud)

In [ ]:
deploy_script = '''\
# deploy.py — corré con: python deploy.py
from prefect import serve
from prefect.client.schemas.schedules import CronSchedule
from your_flow_module import daily_retrain

if __name__ == "__main__":
    deployment = daily_retrain.to_deployment(
        name="daily-retrain-prod",
        schedule=CronSchedule(cron="0 2 * * *", timezone="UTC"),  # 02:00 diario
        parameters={"execution_date": "{{ run.scheduled_start_time | date }}"},
        tags=["prod", "ml-retrain"],
    )
    serve(deployment)
'''
print(deploy_script)

## Ejercicio guiado

1. Agregá una tarea `check_drift` al inicio que computa PSI vs un baseline. Si <0.1: `return` temprano (skip).
2. Convertí `promote_if_better` a `set alias '@challenger'` en vez de `@champion` — la promoción a champion la decidís manualmente o tras shadow OK (Clase 204).
3. Agregá `on_failure_callback` que postea a un Slack webhook (stub si no tenés webhook real).
4. Implementá backfill: corré el flow para `[2026-06-01, ..., 2026-06-07]` y verificá idempotencia.

## Conclusiones

- DAG + champion-challenger = retraining sin regresiones.
- Idempotencia es responsabilidad de cada tarea (insert con upsert, sobrescribir destinos).
- Schedule fijo + trigger por drift se complementan; uno solo deja gaps.
- Promoción auto solo si el delta justifica + shadow OK (Clase 204).